In [0]:
log_data = [
    ("INFO", "2026-07-01 09:15:23"),
    ("WARN", "2026-06-01 09:16:02"),
    ("ERROR", "2026-06-01 09:16:45"),
    ("DEBUG", "2026-05-01 09:17:10"),
    ("INFO", "2026-05-01 09:18:30"),
    ("ERROR", "2026-04-01 09:19:12"),
    ("WARN", "2026-04-01 09:20:05"),
    ("DEBUG", "2026-03-01 09:21:40"),
    ("INFO", "2026-02-01 09:22:18"),
    ("CRITICAL", "2026-01-01 09:23:55")
]

In [0]:
log_df = spark.createDataFrame(log_data).toDF("log_level", "log_time")

In [0]:
log_df.show()

In [0]:
log_df.printSchema()

In [0]:
from pyspark.sql.functions import *

In [0]:
log_df = log_df.withColumn("log_time",to_timestamp("log_time"))

In [0]:
log_df.printSchema()

In [0]:
log_df.createOrReplaceTempView("serverlogs")

In [0]:
spark.sql("select * from serverlogs").show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMMM' ) as month from serverlogs""").show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MM' ) as month from serverlogs""").show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month from serverlogs""").show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'M' ) as month from serverlogs""").show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month, count(*) as total_occurence from serverlogs
          group by log_level,month
          """).show()

In [0]:
log_schema = "log_level string, log_time timestamp"

In [0]:
full_log_df = spark.read.format("csv").schema(log_schema).load('abfss://unitycatalog@databricksstudysa.dfs.core.windows.net/catalog/logdata1m.csv')

In [0]:
full_log_df.show()

In [0]:
full_log_df.count()

In [0]:
full_log_df.createOrReplaceTempView("server_logs")

In [0]:
spark.sql("select * from server_logs").show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month from server_logs
          """).show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month, count(*) as total_occurence from server_logs
          group by log_level,month
          """).show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month, count(*) as total_occurence from server_logs
          group by log_level,month
          order by month
          """).show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month, date_format(log_time,'M') as month_num, count(*) as total_occurence from server_logs
          group by log_level,month, month_num
          order by month_num
          """).show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month, int(date_format(log_time,'M')) as month_num, count(*) as total_occurence from server_logs
          group by log_level,month, month_num
          order by month_num
          """).show(100)

In [0]:
result_df = spark.sql("""
          select log_level, date_format(log_time,'MMM' ) as month, int(date_format(log_time,'M')) as month_num, count(*) as total_occurence from server_logs
          group by log_level,month, month_num
          order by month_num
          """)

In [0]:
from pyspark.sql.functions import *

In [0]:
final_df = result_df.drop("month_num")

In [0]:
final_df.show()

Let's start portray the results in Pivot table

In [0]:
spark.sql("""
          select log_level, date_format(log_time,"MM") as month from server_logs
          """).groupBy("log_level").pivot("month").count().show()

In [0]:
spark.sql("""
          select log_level, date_format(log_time,"MMM") as month from server_logs
          """).groupBy("log_level").pivot("month").count().show()